# NLI Solution A Submission Notebook

This notebook implements **Solution A only** for the NLI track: a traditional machine-learning system within Category A.

Workflow:

1. check or install the required Python packages
2. locate the coursework files safely
3. load the NLI train, dev, and trial data
4. build an internal baseline: concatenated TF-IDF + Logistic Regression
5. build the final Solution A model with richer pairwise features
6. evaluate on the dev set with scorer-aligned metrics
7. compare against the bundled official Category A baseline (`SVM` from `25_DEV_NLI.csv`)
8. save the trained artefacts
9. run demo / inference mode on a trial or test CSV and write a one-column prediction file

This notebook uses only the provided coursework data plus the bundled baseline table for comparison.


In [ ]:
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'scipy': 'scipy',
    'sklearn': 'scikit-learn',
    'joblib': 'joblib',
}

missing_packages = [
    pip_name
    for module_name, pip_name in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages:
    print('Installing missing packages:', missing_packages)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing_packages])
else:
    print('All required packages are already available.')


## 1. Imports and path resolution

The notebook should work whether it is run from the repository root or from inside `solution-a/`.

The path helper below searches upward until it finds the coursework root that contains both:

- `training_data/NLI/train.csv`
- `nlu_bundle-feature-unified-local-scorer/`

This avoids the broken relative-path problem that the earlier notebook had.


In [ ]:
from __future__ import annotations

import re
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler

SEED = 42


def find_project_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (
            (candidate / 'training_data' / 'NLI' / 'train.csv').exists()
            and (candidate / 'nlu_bundle-feature-unified-local-scorer').exists()
        ):
            return candidate
    raise FileNotFoundError(
        'Could not find the coursework root. Expected to find training_data/NLI/train.csv '
        'and nlu_bundle-feature-unified-local-scorer/ in the same project tree.'
    )


PROJECT_ROOT = find_project_root()
NOTEBOOK_DIR = PROJECT_ROOT / 'solution-a'
TRAIN_PATH = PROJECT_ROOT / 'training_data' / 'NLI' / 'train.csv'
DEV_PATH = PROJECT_ROOT / 'training_data' / 'NLI' / 'dev.csv'
TRIAL_PATH = PROJECT_ROOT / 'trial_data' / 'NLI_trial.csv'
LOCAL_SCORER_ROOT = PROJECT_ROOT / 'nlu_bundle-feature-unified-local-scorer'
OFFICIAL_BASELINE_PATH = LOCAL_SCORER_ROOT / 'baseline' / '25_DEV_NLI.csv'
ARTEFACT_DIR = NOTEBOOK_DIR / 'artifacts_solution_a'
ARTEFACT_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('PROJECT_ROOT =', PROJECT_ROOT)
print('TRAIN_PATH   =', TRAIN_PATH)
print('DEV_PATH     =', DEV_PATH)
print('TRIAL_PATH   =', TRIAL_PATH)
print('ARTEFACT_DIR =', ARTEFACT_DIR)


## 2. Data loading

The helper below accepts both training / dev CSVs and trial / test CSVs.

Rules used here:

- `premise` and `hypothesis` are required
- `label` is required only when `require_label=True`
- the helper strips a UTF-8 BOM from the first column name, which is useful for the supplied trial file


In [ ]:
def read_pair_dataframe(csv_path: Path, require_label: bool) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df.columns = [str(col).lstrip('﻿').strip() for col in df.columns]

    required_columns = {'premise', 'hypothesis'}
    missing_required = required_columns - set(df.columns)
    if missing_required:
        raise ValueError(f'{csv_path} is missing required columns: {sorted(missing_required)}')

    if require_label and 'label' not in df.columns:
        raise ValueError(f'{csv_path} must contain a label column for this section of the notebook.')

    if 'label' in df.columns:
        df['label'] = df['label'].astype(int)

    return df


train_df = read_pair_dataframe(TRAIN_PATH, require_label=True)
dev_df = read_pair_dataframe(DEV_PATH, require_label=True)
trial_df = read_pair_dataframe(TRIAL_PATH, require_label=True) if TRIAL_PATH.exists() else None

print('Train rows:', len(train_df))
print('Dev rows:  ', len(dev_df))
print('Trial rows:', len(trial_df) if trial_df is not None else 'trial file not found')
print()
print('Train label distribution:')
print(train_df['label'].value_counts().sort_index())
print()
train_df.head(3)


## 3. Evaluation helpers and official baseline table

The local scorer ships with the following metric names for NLI dev evaluation:

- `accuracy_score`
- `macro_precision`
- `macro_recall`
- `macro_f1`
- `weighted_macro_precision`
- `weighted_macro_recall`
- `weighted_mmacro_f1`
- `matthews_corrcoef`

In this notebook, **model selection is driven by `macro_f1`** rather than plain accuracy.
`macro_f1` gives equal weight to both labels, so it is a better fit than plain accuracy when we want balanced performance.

For Category A comparison, we also load the bundled `SVM` predictions from `25_DEV_NLI.csv`.


In [ ]:
METRIC_ORDER = [
    'accuracy_score',
    'macro_precision',
    'macro_recall',
    'macro_f1',
    'weighted_macro_precision',
    'weighted_macro_recall',
    'weighted_mmacro_f1',
    'matthews_corrcoef',
]


def metric_summary(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    return {
        'accuracy_score': accuracy_score(y_true, y_pred),
        'macro_precision': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'macro_recall': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'weighted_macro_precision': precision_score(y_true, y_pred, average='weighted', zero_division=0),
        'weighted_macro_recall': recall_score(y_true, y_pred, average='weighted', zero_division=0),
        'weighted_mmacro_f1': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'matthews_corrcoef': matthews_corrcoef(y_true, y_pred),
    }


def metrics_table(predictions: dict[str, np.ndarray], y_true: np.ndarray) -> pd.DataFrame:
    rows = []
    for model_name, y_pred in predictions.items():
        row = {'model': model_name}
        row.update(metric_summary(y_true, y_pred))
        rows.append(row)
    df = pd.DataFrame(rows).set_index('model')
    return df[METRIC_ORDER].sort_values('macro_f1', ascending=False)


def mcnemar_test(y_true: np.ndarray, pred_a: np.ndarray, pred_b: np.ndarray, name_a: str, name_b: str) -> dict[str, float | str]:
    correct_a = pred_a == y_true
    correct_b = pred_b == y_true
    b = int(np.sum(correct_a & ~correct_b))
    c = int(np.sum(~correct_a & correct_b))

    if b + c == 0:
        chi2_stat = 0.0
        p_value = 1.0
    else:
        from scipy.stats import chi2
        chi2_stat = (abs(b - c) - 1) ** 2 / (b + c)
        p_value = 1.0 - chi2.cdf(chi2_stat, df=1)

    return {
        'comparison': f'{name_a} vs {name_b}',
        'a_correct_b_wrong': b,
        'a_wrong_b_correct': c,
        'chi2': chi2_stat,
        'p_value': p_value,
    }


official_baseline_df = pd.read_csv(OFFICIAL_BASELINE_PATH)
official_baseline_df = official_baseline_df.rename(columns=lambda col: str(col).strip())
if 'Unnamed: 0' in official_baseline_df.columns:
    official_baseline_df = official_baseline_df.drop(columns=['Unnamed: 0'])

official_baseline_df['reference'] = official_baseline_df['reference'].astype(int)
if official_baseline_df['reference'].tolist() != dev_df['label'].tolist():
    raise ValueError('The reference column in 25_DEV_NLI.csv does not match training_data/NLI/dev.csv.')

official_svm_pred = official_baseline_df['SVM'].astype(int).to_numpy()
y_dev = dev_df['label'].to_numpy(dtype=int)
y_train = train_df['label'].to_numpy(dtype=int)

print('Official baseline methods available:', [col for col in official_baseline_df.columns if col != 'reference'])
pd.DataFrame({'reference': official_baseline_df['reference'].head(5), 'SVM': official_baseline_df['SVM'].head(5)})


## 4. Internal baseline for this notebook

This is the simple baseline used inside this notebook:

- concatenate `premise` and `hypothesis` with a separator token
- represent the concatenated text with word 1-2 gram TF-IDF
- train a Logistic Regression classifier

This baseline is mainly a sanity check. For the coursework comparison, the more important reference point is the **bundled official Category A baseline** (`SVM`).


In [ ]:
baseline_train_text = train_df['premise'].astype(str) + ' [SEP] ' + train_df['hypothesis'].astype(str)
baseline_dev_text = dev_df['premise'].astype(str) + ' [SEP] ' + dev_df['hypothesis'].astype(str)

baseline_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_features=30000,
    sublinear_tf=True,
)
X_baseline_train = baseline_vectorizer.fit_transform(baseline_train_text)
X_baseline_dev = baseline_vectorizer.transform(baseline_dev_text)

internal_baseline_model = LogisticRegression(
    max_iter=2000,
    solver='liblinear',
    random_state=SEED,
)
internal_baseline_model.fit(X_baseline_train, y_train)
internal_baseline_pred = internal_baseline_model.predict(X_baseline_dev)
internal_baseline_metrics = metric_summary(y_dev, internal_baseline_pred)

pd.DataFrame([internal_baseline_metrics], index=['internal_baseline']).T


## 5. Final Solution A design

The final Solution A model stays inside traditional machine learning, but it is stronger than the internal baseline because it adds richer pairwise structure.

Feature blocks used by the final model:

1. **Premise word TF-IDF**: word 1-2 grams from the premise only
2. **Hypothesis word TF-IDF**: word 1-2 grams from the hypothesis only
3. **Shared-space interactions**: absolute difference and element-wise product after projecting both texts into the same word-TF-IDF space
4. **Pair-level character TF-IDF**: character n-grams from `premise [SEP] hypothesis`
5. **Hand-crafted pair features** computed directly from the provided text:
   - lexical overlap
   - new-token ratio
   - length features
   - negation mismatch
   - number mismatch
   - simple punctuation cues

The classifier is still Logistic Regression, but its hyperparameters are tuned with **macro F1** on the training split via cross-validation.


In [ ]:
TOKEN_PATTERN = re.compile(r"[a-z0-9]+(?:'[a-z0-9]+)?")
NUMBER_PATTERN = re.compile(r'\d+(?:\.\d+)?')
NEGATION_TOKENS = {'no', 'not', 'never', 'none', 'nobody', 'nothing', 'neither', 'nor', 'without'}


def normalize_text(text: str) -> str:
    text = str(text)
    text = text.replace('’', "'").replace('‘', "'")
    text = text.replace('“', '"').replace('”', '"')
    return re.sub(r'\s+', ' ', text).strip()


def tokenize(text: str) -> list[str]:
    return TOKEN_PATTERN.findall(normalize_text(text).lower())


HANDCRAFTED_FEATURE_NAMES = [
    'hypothesis_token_recall',
    'premise_token_precision',
    'jaccard',
    'new_token_ratio',
    'premise_length',
    'hypothesis_length',
    'length_ratio',
    'length_difference',
    'premise_has_negation',
    'hypothesis_has_negation',
    'negation_mismatch',
    'shared_number_count',
    'number_mismatch',
    'exact_string_match',
    'hypothesis_token_subset',
    'premise_has_question_mark',
    'hypothesis_has_question_mark',
    'question_mark_delta',
    'premise_has_exclamation_mark',
    'hypothesis_has_exclamation_mark',
    'exclamation_mark_delta',
]


def build_handcrafted_pair_features(df: pd.DataFrame) -> np.ndarray:
    rows = []
    for premise, hypothesis in zip(df['premise'], df['hypothesis']):
        premise_text = normalize_text(premise)
        hypothesis_text = normalize_text(hypothesis)
        premise_tokens = tokenize(premise_text)
        hypothesis_tokens = tokenize(hypothesis_text)
        premise_set = set(premise_tokens)
        hypothesis_set = set(hypothesis_tokens)

        overlap = len(premise_set & hypothesis_set)
        union = len(premise_set | hypothesis_set)
        hypothesis_token_recall = overlap / len(hypothesis_set) if hypothesis_set else 0.0
        premise_token_precision = overlap / len(premise_set) if premise_set else 0.0
        jaccard = overlap / union if union else 0.0
        new_token_ratio = len(hypothesis_set - premise_set) / len(hypothesis_set) if hypothesis_set else 0.0

        premise_length = len(premise_tokens)
        hypothesis_length = len(hypothesis_tokens)
        length_ratio = hypothesis_length / premise_length if premise_length else 0.0
        length_difference = premise_length - hypothesis_length

        premise_has_negation = int(any(tok in NEGATION_TOKENS or tok.endswith("n't") for tok in premise_tokens))
        hypothesis_has_negation = int(any(tok in NEGATION_TOKENS or tok.endswith("n't") for tok in hypothesis_tokens))
        negation_mismatch = int(premise_has_negation != hypothesis_has_negation)

        premise_numbers = NUMBER_PATTERN.findall(premise_text)
        hypothesis_numbers = NUMBER_PATTERN.findall(hypothesis_text)
        shared_number_count = len(set(premise_numbers) & set(hypothesis_numbers))
        number_mismatch = int(bool(premise_numbers or hypothesis_numbers) and set(premise_numbers) != set(hypothesis_numbers))

        exact_string_match = int(premise_text.lower() == hypothesis_text.lower())
        hypothesis_token_subset = int(hypothesis_set.issubset(premise_set)) if hypothesis_set else 0

        premise_has_question_mark = int('?' in premise_text)
        hypothesis_has_question_mark = int('?' in hypothesis_text)
        question_mark_delta = hypothesis_has_question_mark - premise_has_question_mark
        premise_has_exclamation_mark = int('!' in premise_text)
        hypothesis_has_exclamation_mark = int('!' in hypothesis_text)
        exclamation_mark_delta = hypothesis_has_exclamation_mark - premise_has_exclamation_mark

        rows.append([
            hypothesis_token_recall,
            premise_token_precision,
            jaccard,
            new_token_ratio,
            premise_length,
            hypothesis_length,
            length_ratio,
            length_difference,
            premise_has_negation,
            hypothesis_has_negation,
            negation_mismatch,
            shared_number_count,
            number_mismatch,
            exact_string_match,
            hypothesis_token_subset,
            premise_has_question_mark,
            hypothesis_has_question_mark,
            question_mark_delta,
            premise_has_exclamation_mark,
            hypothesis_has_exclamation_mark,
            exclamation_mark_delta,
        ])

    return np.asarray(rows, dtype=np.float32)


def sparse_absolute_difference(left: sp.csr_matrix, right: sp.csr_matrix) -> sp.csr_matrix:
    diff = (left - right).tocsr(copy=True)
    diff.data = np.abs(diff.data)
    return diff


In [ ]:
class SolutionAFeatureBuilder:
    def __init__(
        self,
        premise_word_max_features: int = 12000,
        hypothesis_word_max_features: int = 12000,
        shared_word_max_features: int = 8000,
        pair_char_max_features: int = 8000,
    ) -> None:
        self.premise_word_vectorizer = TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            max_features=premise_word_max_features,
            sublinear_tf=True,
        )
        self.hypothesis_word_vectorizer = TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            max_features=hypothesis_word_max_features,
            sublinear_tf=True,
        )
        self.shared_word_vectorizer = TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            max_features=shared_word_max_features,
            sublinear_tf=True,
        )
        self.pair_char_vectorizer = TfidfVectorizer(
            analyzer='char_wb',
            ngram_range=(3, 5),
            min_df=2,
            max_features=pair_char_max_features,
            sublinear_tf=True,
        )
        self.handcrafted_scaler = StandardScaler()
        self.handcrafted_feature_names_ = HANDCRAFTED_FEATURE_NAMES.copy()
        self.feature_block_dimensions_ = {}

    def _normalised_premise_series(self, df: pd.DataFrame) -> pd.Series:
        return df['premise'].fillna('').map(normalize_text)

    def _normalised_hypothesis_series(self, df: pd.DataFrame) -> pd.Series:
        return df['hypothesis'].fillna('').map(normalize_text)

    def fit(self, df: pd.DataFrame) -> 'SolutionAFeatureBuilder':
        premise_text = self._normalised_premise_series(df)
        hypothesis_text = self._normalised_hypothesis_series(df)
        pair_text = premise_text + ' [SEP] ' + hypothesis_text
        handcrafted = build_handcrafted_pair_features(df)

        self.premise_word_vectorizer.fit(premise_text)
        self.hypothesis_word_vectorizer.fit(hypothesis_text)
        self.shared_word_vectorizer.fit(pd.concat([premise_text, hypothesis_text], ignore_index=True))
        self.pair_char_vectorizer.fit(pair_text)
        self.handcrafted_scaler.fit(handcrafted)

        self.feature_block_dimensions_ = {
            'premise_word_tfidf': len(self.premise_word_vectorizer.get_feature_names_out()),
            'hypothesis_word_tfidf': len(self.hypothesis_word_vectorizer.get_feature_names_out()),
            'shared_abs_difference': len(self.shared_word_vectorizer.get_feature_names_out()),
            'shared_product': len(self.shared_word_vectorizer.get_feature_names_out()),
            'pair_char_tfidf': len(self.pair_char_vectorizer.get_feature_names_out()),
            'handcrafted_dense': len(self.handcrafted_feature_names_),
        }
        return self

    def transform(self, df: pd.DataFrame) -> sp.csr_matrix:
        premise_text = self._normalised_premise_series(df)
        hypothesis_text = self._normalised_hypothesis_series(df)
        pair_text = premise_text + ' [SEP] ' + hypothesis_text

        premise_word = self.premise_word_vectorizer.transform(premise_text)
        hypothesis_word = self.hypothesis_word_vectorizer.transform(hypothesis_text)

        shared_premise = self.shared_word_vectorizer.transform(premise_text)
        shared_hypothesis = self.shared_word_vectorizer.transform(hypothesis_text)
        shared_abs_difference = sparse_absolute_difference(shared_premise, shared_hypothesis)
        shared_product = shared_premise.multiply(shared_hypothesis)

        pair_char = self.pair_char_vectorizer.transform(pair_text)
        handcrafted = build_handcrafted_pair_features(df)
        handcrafted_scaled = self.handcrafted_scaler.transform(handcrafted)

        return sp.hstack(
            [
                premise_word,
                hypothesis_word,
                shared_abs_difference,
                shared_product,
                pair_char,
                sp.csr_matrix(handcrafted_scaled),
            ],
            format='csr',
        )

    def fit_transform(self, df: pd.DataFrame) -> sp.csr_matrix:
        self.fit(df)
        return self.transform(df)


def feature_components_from_builder(feature_builder: SolutionAFeatureBuilder) -> dict:
    return {
        'premise_word_vectorizer': feature_builder.premise_word_vectorizer,
        'hypothesis_word_vectorizer': feature_builder.hypothesis_word_vectorizer,
        'shared_word_vectorizer': feature_builder.shared_word_vectorizer,
        'pair_char_vectorizer': feature_builder.pair_char_vectorizer,
        'handcrafted_scaler': feature_builder.handcrafted_scaler,
        'handcrafted_feature_names': feature_builder.handcrafted_feature_names_,
        'feature_block_dimensions': feature_builder.feature_block_dimensions_,
    }


def transform_with_feature_components(feature_components: dict, df: pd.DataFrame) -> sp.csr_matrix:
    premise_text = df['premise'].fillna('').map(normalize_text)
    hypothesis_text = df['hypothesis'].fillna('').map(normalize_text)
    pair_text = premise_text + ' [SEP] ' + hypothesis_text

    premise_word = feature_components['premise_word_vectorizer'].transform(premise_text)
    hypothesis_word = feature_components['hypothesis_word_vectorizer'].transform(hypothesis_text)

    shared_premise = feature_components['shared_word_vectorizer'].transform(premise_text)
    shared_hypothesis = feature_components['shared_word_vectorizer'].transform(hypothesis_text)
    shared_abs_difference = sparse_absolute_difference(shared_premise, shared_hypothesis)
    shared_product = shared_premise.multiply(shared_hypothesis)

    pair_char = feature_components['pair_char_vectorizer'].transform(pair_text)
    handcrafted = build_handcrafted_pair_features(df)
    handcrafted_scaled = feature_components['handcrafted_scaler'].transform(handcrafted)

    return sp.hstack(
        [
            premise_word,
            hypothesis_word,
            shared_abs_difference,
            shared_product,
            pair_char,
            sp.csr_matrix(handcrafted_scaled),
        ],
        format='csr',
    )


In [ ]:
feature_builder = SolutionAFeatureBuilder()
X_solution_a_train = feature_builder.fit_transform(train_df)
X_solution_a_dev = feature_builder.transform(dev_df)
solution_a_feature_components = feature_components_from_builder(feature_builder)

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
param_grid = {
    'C': [0.5, 1.0, 2.0, 4.0],
    'class_weight': [None, 'balanced'],
}

search = GridSearchCV(
    estimator=LogisticRegression(
        max_iter=2000,
        solver='liblinear',
        random_state=SEED,
    ),
    param_grid=param_grid,
    scoring='f1_macro',
    cv=cv,
    n_jobs=1,
    refit=True,
)
search.fit(X_solution_a_train, y_train)

solution_a_model = search.best_estimator_
solution_a_dev_pred = solution_a_model.predict(X_solution_a_dev)
solution_a_metrics = metric_summary(y_dev, solution_a_dev_pred)

print('Best hyperparameters:', search.best_params_)
print('Best CV macro F1:', round(search.best_score_, 4))
print()
pd.DataFrame([solution_a_metrics], index=['solution_a_final']).T


## 6. Dev-set comparison

This table compares three things:

- the **official Category A baseline** from the bundled scorer table (`SVM`)
- the simple internal notebook baseline
- the final Solution A model

The McNemar test below checks whether the improvement over the same-category baseline is statistically significant.


In [ ]:
comparison_df = metrics_table(
    {
        'Solution A final model': solution_a_dev_pred,
        'Official Category A baseline (SVM)': official_svm_pred,
        'Internal baseline (concat TF-IDF + LR)': internal_baseline_pred,
    },
    y_dev,
)
comparison_df


In [ ]:
significance_df = pd.DataFrame(
    [
        mcnemar_test(y_dev, solution_a_dev_pred, official_svm_pred, 'Solution A final model', 'Official SVM baseline'),
        mcnemar_test(y_dev, solution_a_dev_pred, internal_baseline_pred, 'Solution A final model', 'Internal LR baseline'),
    ]
).set_index('comparison')

significance_df


## 7. Brief error analysis

The goal here is not to make a perfect linguistic analysis inside the notebook.
Instead, we produce a compact, reproducible slice of examples that is useful later for the poster and flash presentation.


In [ ]:
analysis_df = dev_df.copy()
analysis_df['official_svm_pred'] = official_svm_pred
analysis_df['internal_baseline_pred'] = internal_baseline_pred
analysis_df['solution_a_pred'] = solution_a_dev_pred
analysis_df['official_svm_correct'] = analysis_df['official_svm_pred'] == analysis_df['label']
analysis_df['solution_a_correct'] = analysis_df['solution_a_pred'] == analysis_df['label']

label_breakdown = analysis_df.groupby('label')['solution_a_correct'].mean().rename('solution_a_accuracy_by_label')
print('Solution A accuracy by gold label:')
print(label_breakdown)
print()

improved_examples = analysis_df[
    analysis_df['solution_a_correct'] & ~analysis_df['official_svm_correct']
][['premise', 'hypothesis', 'label', 'official_svm_pred', 'solution_a_pred']].head(5)

remaining_errors = analysis_df[
    ~analysis_df['solution_a_correct']
][['premise', 'hypothesis', 'label', 'official_svm_pred', 'solution_a_pred']].head(5)

print('Examples where Solution A is correct and the official SVM baseline is wrong:')
improved_examples


In [ ]:
print('Examples where Solution A is still wrong:')
remaining_errors


## 8. Save the trained artefacts

The saved bundle is what the demo / inference section will load.

Files written here:

- `nli_solution_a_bundle.joblib`: model + saved feature components + metadata
- `nli_solution_a_dev_predictions.csv`: one-column dev predictions, scorer-friendly
- `nli_solution_a_dev_metrics.csv`: metric summary table for quick reference




In [ ]:
SOLUTION_A_BUNDLE_PATH = ARTEFACT_DIR / 'nli_solution_a_bundle.joblib'
SOLUTION_A_DEV_PRED_PATH = ARTEFACT_DIR / 'nli_solution_a_dev_predictions.csv'
SOLUTION_A_METRICS_PATH = ARTEFACT_DIR / 'nli_solution_a_dev_metrics.csv'

solution_a_bundle = {
    'model': solution_a_model,
    'feature_components': solution_a_feature_components,
    'metadata': {
        'solution_name': 'NLI Solution A',
        'category': 'A',
        'task': 'NLI',
        'best_params': search.best_params_,
        'best_cv_macro_f1': float(search.best_score_),
        'feature_block_dimensions': solution_a_feature_components['feature_block_dimensions'],
        'handcrafted_feature_names': solution_a_feature_components['handcrafted_feature_names'],
        'notes': 'Traditional machine-learning solution using TF-IDF interaction features and text-derived pair features only.',
    },
}

joblib.dump(solution_a_bundle, SOLUTION_A_BUNDLE_PATH)
pd.Series(solution_a_dev_pred, name='label').to_csv(SOLUTION_A_DEV_PRED_PATH, index=False, header=False)
comparison_df.to_csv(SOLUTION_A_METRICS_PATH)

print('Saved:', SOLUTION_A_BUNDLE_PATH)
print('Saved:', SOLUTION_A_DEV_PRED_PATH)
print('Saved:', SOLUTION_A_METRICS_PATH)


## 9. Demo / inference mode

This section is intentionally separated from training.

Use it after the artefact bundle exists. It accepts a CSV that contains at least:

- `premise`
- `hypothesis`

If the input also contains `label` (for example the trial file), the helper prints a quick sanity-check metric summary.

For final coursework submission, rename the final test predictions to the required format:

- `Group_n_A.csv`


In [ ]:
def load_solution_a_bundle(bundle_path: Path = SOLUTION_A_BUNDLE_PATH) -> dict:
    return joblib.load(bundle_path)


def predict_with_solution_a(
    input_csv: Path,
    output_csv: Path,
    bundle_path: Path = SOLUTION_A_BUNDLE_PATH,
) -> tuple[pd.DataFrame, np.ndarray, Path]:
    bundle = load_solution_a_bundle(bundle_path)
    input_df = read_pair_dataframe(Path(input_csv), require_label=False)
    X_input = transform_with_feature_components(bundle['feature_components'], input_df)
    predictions = bundle['model'].predict(X_input).astype(int)

    output_csv = Path(output_csv)
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    pd.Series(predictions, name='label').to_csv(output_csv, index=False, header=False)

    return input_df, predictions, output_csv


In [ ]:
DEMO_INPUT_PATH = TRIAL_PATH if TRIAL_PATH.exists() else None
DEMO_OUTPUT_PATH = ARTEFACT_DIR / 'nli_solution_a_trial_predictions.csv'

if DEMO_INPUT_PATH is None:
    print('No trial or test file was found automatically. Set DEMO_INPUT_PATH to a CSV with premise and hypothesis columns.')
else:
    demo_df, demo_predictions, demo_output_path = predict_with_solution_a(
        input_csv=DEMO_INPUT_PATH,
        output_csv=DEMO_OUTPUT_PATH,
    )
    print('Wrote demo predictions to:', demo_output_path)
    print('Number of predictions:', len(demo_predictions))

    if 'label' in demo_df.columns:
        print()
        print('Trial-file sanity-check metrics:')
        print(pd.Series(metric_summary(demo_df['label'].to_numpy(dtype=int), demo_predictions)))
